In [1]:
import tensorflow as tf
from sklearn.model_selection import train_test_split

# 1. Load the native 2-way split
(x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# 2. Carve out the Validation set from the Training set
# We will take 10,000 images for validation, leaving 50,000 for training
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full,
    test_size=10000,
    random_state=42
)

# 3. Normalize the pixels (Crucial step!)
x_train = x_train.astype('float32') / 255.0
x_val = x_val.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# 4. Reshape for a Convolutional Neural Network (Adding the color channel)
x_train = x_train.reshape(-1, 28, 28, 1)
x_val = x_val.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

# 5. Build the tf.data Pipelines
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_dataset = train_dataset.shuffle(10000).batch(32).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((x_val, y_val))
val_dataset = val_dataset.batch(32).prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((x_test, y_test))
test_dataset = test_dataset.batch(32).prefetch(tf.data.AUTOTUNE)

In [2]:
from functools import partial

DefaultConv2D=partial(tf.keras.layers.Conv2D,kernel_size=3,padding='same',
activation='relu',kernel_initializer='he_normal')

model=tf.keras.Sequential([
    DefaultConv2D(filters=64,kernel_size=7,input_shape=[28,28,1]),
    tf.keras.layers.MaxPool2D(),
    DefaultConv2D(filters=128),
    DefaultConv2D(filters=128),
    tf.keras.layers.MaxPool2D(),
    DefaultConv2D(filters=256),
    DefaultConv2D(filters=256),
    tf.keras.layers.MaxPool2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(units=128,activation='relu',
    kernel_initializer='he_normal'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(units=64,activation='relu',
    kernel_initializer='he_normal'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(units=10,activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [3]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="adam",
                metrics=["accuracy"])

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',  # What to watch (default is 'val_loss')
    patience=10,         # How many epochs to wait (passed as a keyword argument)
    restore_best_weights=True
)

history=model.fit(x_train,y_train,epochs=1000,
                  validation_data=(x_val,y_val),
                  callbacks=[early_stopping])

test_loss, test_accuracy = model.evaluate(x_test, y_test)
print(f"Test Accuracy: {test_accuracy:.4f}")

Epoch 1/1000
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 30s 12ms/step - accuracy: 0.6206 - loss: 1.0918 - val_accuracy: 0.9812 - val_loss: 0.0815
Epoch 2/1000
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - accuracy: 0.9577 - loss: 0.1774 - val_accuracy: 0.9857 - val_loss: 0.0672
Epoch 3/1000
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - accuracy: 0.9739 - loss: 0.1119 - val_accuracy: 0.9867 - val_loss: 0.0558
Epoch 4/1000
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - accuracy: 0.9815 - loss: 0.0793 - val_accuracy: 0.9893 - val_loss: 0.0528
Epoch 5/1000
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - accuracy: 0.9822 - loss: 0.0750 - val_accuracy: 0.9911 - val_loss: 0.0412
Epoch 6/1000
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - accuracy: 0.9870 - loss: 0.0567 - val_accuracy: 0.9887 - val_loss: 0.0520
Epoch 7/1000
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - accuracy: 0.9850 - loss: 0.0661 - val_accuracy: 0.9914 - val_loss: 0.0477
Epoch 8/1000
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - accuracy:

In [4]:
import numpy as np

# 1. Load the native 2-way split
(x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# 2. Combine ALL data (60,000 train + 10,000 test = 70,000 total)
x_all = np.concatenate([x_train_full, x_test], axis=0)
y_all = np.concatenate([y_train_full, y_test], axis=0)

# 3. Normalize
x_all = x_all.astype('float32') / 255.0

# 4. Reshape for CNN
x_all = x_all.reshape(-1, 28, 28, 1)

# 5. Build the final training pipeline (no validation split)
final_dataset = tf.data.Dataset.from_tensor_slices((x_all, y_all))
final_dataset = final_dataset.shuffle(70000).batch(32).prefetch(tf.data.AUTOTUNE)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',  # What to watch (default is 'val_loss')
    patience=10,         # How many epochs to wait (passed as a keyword argument)
    restore_best_weights=True
)

final_model=model.fit(x_all,y_all,epochs=16,
                  callbacks=[early_stopping])

Epoch 1/16
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 28s 9ms/step - accuracy: 0.6498 - loss: 1.0051
Epoch 2/16
  11/2188 ━━━━━━━━━━━━━━━━━━━━ 25s 12ms/step - accuracy: 0.9774 - loss: 0.0902

/usr/local/lib/python3.12/dist-packages/keras/src/callbacks/early_stopping.py:153: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)


2188/2188 ━━━━━━━━━━━━━━━━━━━━ 16s 7ms/step - accuracy: 0.9679 - loss: 0.1293
Epoch 3/16
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 15s 7ms/step - accuracy: 0.9776 - loss: 0.0949
Epoch 4/16
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 15s 7ms/step - accuracy: 0.9833 - loss: 0.0728
Epoch 5/16
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - accuracy: 0.9842 - loss: 0.0692
Epoch 6/16
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 15s 7ms/step - accuracy: 0.9866 - loss: 0.0559
Epoch 7/16
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 15s 7ms/step - accuracy: 0.9889 - loss: 0.0482
Epoch 8/16
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 15s 7ms/step - accuracy: 0.9898 - loss: 0.0431
Epoch 9/16
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 15s 7ms/step - accuracy: 0.9906 - loss: 0.0401
Epoch 10/16
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 15s 7ms/step - accuracy: 0.9924 - loss: 0.0363
Epoch 11/16
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 15s 7ms/step - accuracy: 0.9913 - loss: 0.0392
Epoch 12/16
2188/2188 ━━━━━━━━━━━━━━━━━━━━ 15s 7ms/step - accuracy: 0.9918 - loss: 0.0358
Epoch 13/16
2188/2188 ━━━━━━━

In [6]:
model.save('my_mnist_model.v2.keras')

print("Model successfully saved!")

Model successfully saved!
